In [1]:
from IPMSP_module import solve_instance, run_qaoa_with_params, is_feasible, plot_gantt_chart, invert_counts, assign_jobs,get_job_assignment,plot_bar_chart
import time
from matplotlib import rcParams

In [2]:
# 定义问题参数
n = 5                      # 作业数（jobs）
m = 3                      # 机器数（machines）
inst_L = [3,3,2,1,1]       # 每个作业的加工时间/长度
n_runs = 1                 # QAOA 随机初始化并求解的次数
# optimizers = ['COBYLA', 'Nelder-Mead', 'Powell', 'BFGS', 'SLSQP']
optimizers = ['Powell']    # 选择使用的经典优化器列表，这里只用 Powell

start = time.process_time()  # 记录起始 CPU 时间，用于计算运行耗时

# -------------------- 参数合法性检查 --------------------
# 判定：对输入的 n, m, inst_L 做一些基本范围检查，避免无意义或错误输入

if m <= 0:
    raise ValueError("车间数目不能小于0")          # 至少要有 1 个车间

if m > 5:
    raise ValueError("车间数目不能大于5")          # 这里给定上限为 5，可按需求调整

if n < m:
    raise ValueError("作业数不能小于车间数目")      # 常见约束：作业数通常不小于车间数

if len(inst_L) != n:
    raise ValueError("作业长度数目要等于作业数目")  # inst_L 中的作业长度个数应与 n 一致

if m * n > 32:
    # 实际约束：比特数 = 作业数 × 机器数，超过一定上限可能导致 QAOA 线路太大
    raise ValueError("比特数(车间数目*作业数目)超过最大值(30)")

# -------------------- 调用 QAOA 优化求解 --------------------
# 调用 solve_instance 函数，在给定问题规模和实例下，
# 使用指定的优化器进行 n_runs 次随机初始化的 QAOA 优化

results, Q, g, c = solve_instance(
    n, 
    m, 
    inst_L, 
    n_runs=n_runs, 
    optimizers=optimizers
)

# -------------------- 打印优化结果 --------------------
print("优化结果：")
for opt in optimizers:
    print(f"Optimizer: {opt}")
    # 对于该优化器的每一次随机运行，打印运行编号、beta、gamma、能量等信息
    for run_info in results[f"N_{n}M_{m}"][opt]:
        print(run_info)

# -------------------- 选取最佳参数 --------------------
# 简单策略：从第一个优化器的所有运行结果中，选能量最小的一组参数 (beta, gamma)
chosen_optimizer = optimizers[0]                             # 这里就是 'Powell'
runs = results[f"N_{n}M_{m}"][chosen_optimizer]              # 该优化器全部 run 的结果列表
best_run = min(runs, key=lambda x: x['energy'])              # 取出能量最小的那一条记录
best_beta = [best_run['beta']]                               # 封装为列表形式，适配后续 QAOA 接口
best_gamma = [best_run['gamma']]

# -------------------- 用最佳参数运行 QAOA 电路并评估 --------------------
# 使用选出的最优 (beta, gamma) 再跑一次 QAOA 电路，
# 得到对应的平均能量和测量分布 counts

avg_energy, counts, Q, g, c = run_qaoa_with_params(
    n, 
    m, 
    inst_L, 
    best_beta, 
    best_gamma, 
    shots=10000            # 采样次数，用于估计期望能量
)

print(f'优化后平均能量 = {avg_energy}')

# -------------------- 经典启发式调度方案（对比用） --------------------
# 使用 assign_jobs 函数（经典方法）生成一个调度方案，
# 作为对 QAOA 求解结果的参考/对照

classical_assignments, classical_bitstring, classical_decimal = assign_jobs(n, m, inst_L)

print("机器分配方案：")
for mm, assign_list in enumerate(classical_assignments):
    # 将该机器上的作业格式化为 J{id}(len={length}) 的形式便于阅读
    jobs_on_machine = [f"J{j_id}(len={length})" for (j_id, length) in assign_list]
    print(f"Machine {mm}: {jobs_on_machine}")

end1 = time.process_time()   # 记录结束时间
print("运行时间:", end1 - start)  # 输出总运行耗时（包含 QAOA + 经典分配等）

run0
随机初始点[1.36993211 0.73850996]
optimizer=Powell
优化结果：
Optimizer: Powell
{'run': 0, 'beta': 1.7938038749234013, 'gamma': 1.8836208397459504, 'energy': -362.46415824284213}
优化后平均能量 = -353.4818
机器分配方案：
Machine 0: ['J0(len=3)', 'J4(len=1)']
Machine 1: ['J1(len=3)']
Machine 2: ['J2(len=2)', 'J3(len=1)']
运行时间: 6.0


In [3]:
# 将量子测量得到的 counts（bitstring -> 计数）按计数从大到小排序，
# 然后再用 invert_counts 把 bitstring 反转，得到新的 sorted_dict
sorted_dict = invert_counts(
    dict(sorted(counts.items(), key=lambda item: item[1], reverse=True))
)

# 建一个字典，key 换成十进制表示，value 保持计数不变
decimal_counts = {}
for item, count in sorted_dict.items():
    # item 是二进制字符串，转换成十进制整数，再转为字符串作为 key
    decimal_number = str(int(item, 2))
    decimal_counts[decimal_number] = count

# 取前 30 个解（按计数排序后的前 30 个）
keys = list(decimal_counts.keys())[:30]      # 十进制解（字符串）
values = list(decimal_counts.values())[:30]  # 对应的计数

# 把这前 30 个十进制解还原回固定长度的二进制串，用于判断可行性
binary_solutions = []
for dec_str in keys:
    dec_val = int(dec_str)                       # 转回 int
    bin_str = bin(dec_val)[2:].zfill(n*m)        # 去掉 '0b'，按 n*m 位补零
    binary_solutions.append(bin_str)

# 颜色列表，用于画柱状图（区分可行解 / 不可行解）
colors = []
# 记录第一个找到的可行解，用于后面画甘特图
feasible_solution_for_gantt = None

# 遍历每个二进制解，检查是否是可行调度（每个作业段恰有一个 '1'）
for bin_str in binary_solutions:
    if is_feasible(bin_str, n, m):
        # 可行解标记为红色
        colors.append('red')
        # 保存第一个可行解，给后面绘制甘特图使用
        if feasible_solution_for_gantt is None:
            feasible_solution_for_gantt = bin_str
    else:
        # 不可行解标记为灰色
        colors.append('grey')

# 确保经典启发式得到的解 classical_decimal 也展示在柱状图中
if str(classical_decimal) not in keys:
    # 如果 classical_decimal 不在前30解中，就把它插到第一个位置
    # 这里借用当前 decimal_counts 的第一个 value 作为一个”占位“计数值
    classical_value = list(decimal_counts.values())[0]
    keys.insert(0, str(classical_decimal))      # 十进制编码插入到开头
    values.insert(0, classical_value)          # 对应的计数插入到开头
    binary_solutions.insert(0, classical_bitstring)  # 插入它对应的二进制解
    colors.append('red')                       # 把它视作可行解，用红色表示
    if feasible_solution_for_gantt is None:
        # 如果之前还没有可行解，就用经典解做甘特图
        feasible_solution_for_gantt = classical_bitstring

# -------------------- 绘图部分 --------------------

# 如果找到了至少一个可行解，则绘制柱状图与甘特图
if feasible_solution_for_gantt is not None:
    # 绘制解的分布柱状图（keys 为十进制解，values 为计数，colors 区分可行性）
    plot_bar_chart(keys, values, colors)
    # 绘制甘特图，这里使用经典启发式的解 classical_bitstring 作为调度方案
    plot_gantt_chart(classical_bitstring, n, m, inst_L)
else:
    # 如果前 30 个解以及 classical 解都不可行，则给出提示
    print("No feasible solution found.")

# 记录并打印从 start 到现在的总运行时间
end2 = time.process_time()
print("运行时间:", end2 - start)


运行时间: 6.375
